An end-to-end fraud analysis on 590,540 real-world payment transactions,
identifying high-risk patterns across time windows, card types, devices,
and transaction amounts. Mirrors real payment fraud monitoring workflows
used at fintech companies.

In [1]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Load data
trans = pd.read_csv('/content/drive/MyDrive/train_transaction.csv')
identity = pd.read_csv('/content/drive/MyDrive/train_identity.csv')

print("Transactions shape:", trans.shape)
print("Identity shape:", identity.shape)
print("\nFraud rate:", round(trans['isFraud'].mean() * 100, 2), "%")

trans.head()

Transactions shape: (590540, 394)
Identity shape: (144233, 41)

Fraud rate: 3.5 %


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [2]:
# Merge transaction + identity on TransactionID
df = trans.merge(identity, on='TransactionID', how='left')
print("Merged shape:", df.shape)

# Keep only the most analytically meaningful columns
cols = [
    'TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
    'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain',
    'DeviceType', 'DeviceInfo'
]
df = df[cols]
print("\nWorking dataframe shape:", df.shape)
print("\nColumn nulls (%):")
print(round(df.isnull().sum() / len(df) * 100, 1))

Merged shape: (590540, 434)

Working dataframe shape: (590540, 17)

Column nulls (%):
TransactionID      0.0
isFraud            0.0
TransactionDT      0.0
TransactionAmt     0.0
ProductCD          0.0
card1              0.0
card2              1.5
card3              0.3
card4              0.3
card5              0.7
card6              0.3
addr1             11.1
addr2             11.1
P_emaildomain     16.0
R_emaildomain     76.8
DeviceType        76.2
DeviceInfo        79.9
dtype: float64


In [3]:
# Convert transaction time (seconds from reference) to usable time features
df['hour'] = (df['TransactionDT'] // 3600) % 24
df['day_of_week'] = (df['TransactionDT'] // (3600 * 24)) % 7  # 0=Mon approx
df['day_num'] = df['TransactionDT'] // (3600 * 24)

# Clean up key categoricals
df['card4'] = df['card4'].fillna('unknown')        # card network: visa/mastercard etc
df['card6'] = df['card6'].fillna('unknown')        # credit vs debit
df['DeviceType'] = df['DeviceType'].fillna('unknown')
df['P_emaildomain'] = df['P_emaildomain'].fillna('unknown')

# Transaction amount bins (like a real analyst would segment)
df['amt_bucket'] = pd.cut(df['TransactionAmt'],
    bins=[0, 25, 50, 100, 200, 500, 1000, 20000],
    labels=['0-25', '25-50', '50-100', '100-200', '200-500', '500-1K', '1K+'])

print("New columns added: hour, day_of_week, day_num, amt_bucket")
print("\nCard network distribution:")
print(df['card4'].value_counts())
print("\nCredit vs Debit:")
print(df['card6'].value_counts())
print("\nDevice type:")
print(df['DeviceType'].value_counts())

New columns added: hour, day_of_week, day_num, amt_bucket

Card network distribution:
card4
visa                384767
mastercard          189217
american express      8328
discover              6651
unknown               1577
Name: count, dtype: int64

Credit vs Debit:
card6
debit              439938
credit             148986
unknown              1571
debit or credit        30
charge card            15
Name: count, dtype: int64

Device type:
DeviceType
unknown    449730
desktop     85165
mobile      55645
Name: count, dtype: int64


In [4]:
# ── 1. FRAUD RATE BY HOUR OF DAY ──────────────────────────────────────────
hourly = df.groupby('hour').agg(
    total=('isFraud', 'count'),
    fraud=('isFraud', 'sum')
).reset_index()
hourly['fraud_rate'] = round(hourly['fraud'] / hourly['total'] * 100, 2)

# ── 2. FRAUD RATE BY DAY OF WEEK ──────────────────────────────────────────
daily = df.groupby('day_of_week').agg(
    total=('isFraud', 'count'),
    fraud=('isFraud', 'sum')
).reset_index()
daily['fraud_rate'] = round(daily['fraud'] / daily['total'] * 100, 2)
daily['day_name'] = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

# ── 3. FRAUD RATE BY CARD NETWORK ─────────────────────────────────────────
card_fraud = df.groupby('card4').agg(
    total=('isFraud', 'count'),
    fraud=('isFraud', 'sum')
).reset_index()
card_fraud['fraud_rate'] = round(card_fraud['fraud'] / card_fraud['total'] * 100, 2)
card_fraud = card_fraud[card_fraud['card4'] != 'unknown'].sort_values('fraud_rate', ascending=False)

# ── 4. FRAUD RATE BY CREDIT VS DEBIT ──────────────────────────────────────
card6_fraud = df[df['card6'].isin(['credit', 'debit'])].groupby('card6').agg(
    total=('isFraud', 'count'),
    fraud=('isFraud', 'sum')
).reset_index()
card6_fraud['fraud_rate'] = round(card6_fraud['fraud'] / card6_fraud['total'] * 100, 2)

# ── 5. FRAUD RATE BY TRANSACTION AMOUNT BUCKET ────────────────────────────
amt_fraud = df.groupby('amt_bucket', observed=True).agg(
    total=('isFraud', 'count'),
    fraud=('isFraud', 'sum')
).reset_index()
amt_fraud['fraud_rate'] = round(amt_fraud['fraud'] / amt_fraud['total'] * 100, 2)

# ── 6. FRAUD RATE BY DEVICE TYPE ──────────────────────────────────────────
device_fraud = df.groupby('DeviceType').agg(
    total=('isFraud', 'count'),
    fraud=('isFraud', 'sum')
).reset_index()
device_fraud['fraud_rate'] = round(device_fraud['fraud'] / device_fraud['total'] * 100, 2)

# ── 7. FRAUD RATE BY EMAIL DOMAIN (top 10) ────────────────────────────────
email_fraud = df[df['P_emaildomain'] != 'unknown'].groupby('P_emaildomain').agg(
    total=('isFraud', 'count'),
    fraud=('isFraud', 'sum')
).reset_index()
email_fraud['fraud_rate'] = round(email_fraud['fraud'] / email_fraud['total'] * 100, 2)
email_fraud = email_fraud[email_fraud['total'] > 1000].sort_values('fraud_rate', ascending=False).head(10)

# ── PRINT SUMMARY ──────────────────────────────────────────────────────────
print("=== FRAUD RATE BY CARD NETWORK ===")
print(card_fraud[['card4', 'total', 'fraud', 'fraud_rate']].to_string(index=False))

print("\n=== FRAUD RATE BY CREDIT VS DEBIT ===")
print(card6_fraud[['card6', 'total', 'fraud', 'fraud_rate']].to_string(index=False))

print("\n=== FRAUD RATE BY DEVICE ===")
print(device_fraud[['DeviceType', 'total', 'fraud', 'fraud_rate']].to_string(index=False))

print("\n=== FRAUD RATE BY AMOUNT BUCKET ===")
print(amt_fraud[['amt_bucket', 'total', 'fraud', 'fraud_rate']].to_string(index=False))

print("\n=== TOP 10 EMAIL DOMAINS BY FRAUD RATE ===")
print(email_fraud[['P_emaildomain', 'total', 'fraud', 'fraud_rate']].to_string(index=False))

print("\n=== PEAK FRAUD HOURS (top 5) ===")
print(hourly.sort_values('fraud_rate', ascending=False).head(5)[['hour', 'total', 'fraud', 'fraud_rate']].to_string(index=False))

=== FRAUD RATE BY CARD NETWORK ===
           card4  total  fraud  fraud_rate
        discover   6651    514        7.73
            visa 384767  13373        3.48
      mastercard 189217   6496        3.43
american express   8328    239        2.87

=== FRAUD RATE BY CREDIT VS DEBIT ===
 card6  total  fraud  fraud_rate
credit 148986   9950        6.68
 debit 439938  10674        2.43

=== FRAUD RATE BY DEVICE ===
DeviceType  total  fraud  fraud_rate
   desktop  85165   5554        6.52
    mobile  55645   5657       10.17
   unknown 449730   9452        2.10

=== FRAUD RATE BY AMOUNT BUCKET ===
amt_bucket  total  fraud  fraud_rate
      0-25  50829   3150        6.20
     25-50 153695   4683        3.05
    50-100 164095   4788        2.92
   100-200 128041   3899        3.05
   200-500  71001   3135        4.42
    500-1K  15612    829        5.31
       1K+   7265    179        2.46

=== TOP 10 EMAIL DOMAINS BY FRAUD RATE ===
P_emaildomain  total  fraud  fraud_rate
  outlook.com   5

In [6]:
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Fraud Rate by Hour of Day',
        'Fraud Rate by Card Network',
        'Fraud Rate by Device Type',
        'Fraud Rate: Credit vs Debit',
        'Fraud Rate by Transaction Amount',
        'Top Email Domains by Fraud Rate'
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

# ── 1. Hourly fraud rate (line) ────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=hourly['hour'], y=hourly['fraud_rate'],
    mode='lines+markers',
    line=dict(color='#E63946', width=2.5),
    marker=dict(size=6),
    name='Hourly Fraud Rate'
), row=1, col=1)

# shade peak hours 5-9
fig.add_vrect(x0=5, x1=9, fillcolor='rgba(230,57,70,0.12)',
              line_width=0, row=1, col=1)

# ── 2. Fraud rate by card network (bar) ───────────────────────────────────
fig.add_trace(go.Bar(
    x=card_fraud['card4'],
    y=card_fraud['fraud_rate'],
    marker_color=['#E63946', '#457B9D', '#457B9D', '#457B9D'],
    name='Card Network'
), row=1, col=2)

# ── 3. Device type (horizontal bar) ───────────────────────────────────────
device_plot = device_fraud.sort_values('fraud_rate')
fig.add_trace(go.Bar(
    x=device_plot['fraud_rate'],
    y=device_plot['DeviceType'],
    orientation='h',
    marker_color=['#A8DADC', '#457B9D', '#E63946'],
    name='Device Type'
), row=2, col=1)

# ── 4. Credit vs Debit ────────────────────────────────────────────────────
fig.add_trace(go.Bar(
    x=card6_fraud['card6'],
    y=card6_fraud['fraud_rate'],
    marker_color=['#E63946', '#457B9D'],
    name='Card Type'
), row=2, col=2)

# ── 5. Amount bucket ──────────────────────────────────────────────────────
fig.add_trace(go.Bar(
    x=amt_fraud['amt_bucket'].astype(str),
    y=amt_fraud['fraud_rate'],
    marker_color=['#E63946' if x in ['0-25', '500-1K'] else '#457B9D'
                  for x in amt_fraud['amt_bucket'].astype(str)],
    name='Amount Bucket'
), row=3, col=1)

# ── 6. Email domain ───────────────────────────────────────────────────────
email_plot = email_fraud.sort_values('fraud_rate')
fig.add_trace(go.Bar(
    x=email_plot['fraud_rate'],
    y=email_plot['P_emaildomain'],
    orientation='h',
    marker_color=['#E63946' if x >= 7 else '#457B9D' if x >= 4 else '#A8DADC'
                  for x in email_plot['fraud_rate']],
    name='Email Domain'
), row=3, col=2)

# ── Layout ─────────────────────────────────────────────────────────────────
fig.update_layout(
    height=1000,
    title_text='IEEE-CIS Fraud Detection — Pattern Analysis (590K Transactions)',
    title_font=dict(size=18, color='#1D3557'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False,
    font=dict(family='Arial', size=11, color='#333333')
)

# Add % suffix to all y-axes that show fraud rate
fig.update_xaxes(title_text='Hour of Day', row=1, col=1)
fig.update_xaxes(title_text='Fraud Rate %', row=2, col=1)
fig.update_xaxes(title_text='Fraud Rate %', row=3, col=2)

# Y-axis % suffix for vertical charts
fig.update_yaxes(ticksuffix='%', row=1, col=1)
fig.update_yaxes(ticksuffix='%', row=1, col=2)
fig.update_yaxes(ticksuffix='%', row=2, col=2)
fig.update_yaxes(ticksuffix='%', row=3, col=1)

# X-axis % suffix for horizontal charts
fig.update_xaxes(ticksuffix='%', row=2, col=1)
fig.update_xaxes(ticksuffix='%', row=3, col=2)

fig.show()

In [7]:
# Pivot: average fraud rate by hour and day of week
heatmap_data = df.groupby(['day_of_week', 'hour'])['isFraud'].mean() * 100
heatmap_pivot = heatmap_data.unstack(level='hour')

day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig2 = go.Figure(data=go.Heatmap(
    z=heatmap_pivot.values,
    x=[f"{h:02d}:00" for h in heatmap_pivot.columns],
    y=day_labels,
    colorscale=[
        [0.0, '#EAF4FB'],
        [0.3, '#457B9D'],
        [0.6, '#E63946'],
        [1.0, '#6B0F1A']
    ],
    colorbar=dict(
        title='Fraud Rate %',
        ticksuffix='%'
    ),
    hoverongaps=False,
    hovertemplate='Day: %{y}<br>Hour: %{x}<br>Fraud Rate: %{z:.2f}%<extra></extra>'
))

fig2.update_layout(
    title=dict(
        text='Fraud Rate Heatmap — Hour of Day × Day of Week',
        font=dict(size=16, color='#1D3557')
    ),
    xaxis=dict(title='Hour of Day', tickangle=-45),
    yaxis=dict(title='Day of Week'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=400,
    font=dict(family='Arial', size=11)
)

fig2.show()

In [8]:
from scipy import stats

# Flag statistical anomalies in transaction amount using Z-score
df['amt_zscore'] = np.abs(stats.zscore(df['TransactionAmt']))
df['is_anomaly'] = df['amt_zscore'] > 3  # beyond 3 standard deviations

# Compare anomaly flag vs actual fraud
anomaly_summary = pd.DataFrame({
    'Category': ['Normal Transactions', 'Statistical Anomalies'],
    'Count': [
        len(df[df['is_anomaly']==False]),
        len(df[df['is_anomaly']==True])
    ],
    'Fraud Rate %': [
        round(df[df['is_anomaly']==False]['isFraud'].mean() * 100, 2),
        round(df[df['is_anomaly']==True]['isFraud'].mean() * 100, 2)
    ]
})

print("=== ANOMALY DETECTION RESULTS ===")
print(anomaly_summary.to_string(index=False))
print(f"\nTotal anomalies flagged: {df['is_anomaly'].sum():,}")
print(f"Anomaly rate: {df['is_anomaly'].mean()*100:.2f}%")

# High risk transaction profile
print("\n=== HIGH RISK TRANSACTION PROFILE ===")
print("Fraud rate by combined risk factors:")

risk = df.copy()
risk['high_risk'] = (
    (risk['hour'].between(5, 9)) &
    (risk['card6'] == 'credit') &
    (risk['DeviceType'] == 'mobile')
).astype(int)

risk_summary = risk.groupby('high_risk')['isFraud'].agg(['mean', 'count']).reset_index()
risk_summary.columns = ['high_risk', 'fraud_rate', 'count']
risk_summary['fraud_rate'] = round(risk_summary['fraud_rate'] * 100, 2)
risk_summary['high_risk'] = risk_summary['high_risk'].map({0: 'Standard', 1: 'High Risk Profile'})
print(risk_summary.to_string(index=False))

=== ANOMALY DETECTION RESULTS ===
             Category  Count  Fraud Rate %
  Normal Transactions 580447          3.47
Statistical Anomalies  10093          5.33

Total anomalies flagged: 10,093
Anomaly rate: 1.71%

=== HIGH RISK TRANSACTION PROFILE ===
Fraud rate by combined risk factors:
        high_risk  fraud_rate  count
         Standard        3.43 588815
High Risk Profile       26.96   1725


In [9]:
# Key findings summary visualization
findings = {
    'Mobile Transactions': 10.17,
    'Early Morning (5-9am)': 9.72,
    'Discover Card': 7.73,
    'Credit Card': 6.68,
    'Micro Transactions (<$25)': 6.20,
    'outlook.com Email': 9.46,
    'High Risk Profile\n(mobile+credit+5-9am)': 26.96,
    'Baseline Fraud Rate': 3.50
}

colors = ['#E63946' if v > 5 else '#457B9D' if v > 3.5 else '#2A9D8F'
          for v in findings.values()]

fig3 = go.Figure(go.Bar(
    x=list(findings.keys()),
    y=list(findings.values()),
    marker_color=colors,
    text=[f"{v}%" for v in findings.values()],
    textposition='outside',
    textfont=dict(size=11, color='#333333')
))

fig3.add_hline(
    y=3.50, line_dash='dash',
    line_color='#2A9D8F', line_width=2,
    annotation_text='Baseline: 3.5%',
    annotation_position='right'
)

fig3.update_layout(
    title=dict(
        text='Key Fraud Risk Factors — Fraud Rate vs 3.5% Baseline',
        font=dict(size=16, color='#1D3557')
    ),
    yaxis=dict(title='Fraud Rate %', ticksuffix='%'),
    xaxis=dict(tickangle=-20),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=500,
    font=dict(family='Arial', size=11),
    showlegend=False
)

fig3.show()